# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. The dataset includes ordered logistic regression outputs regarding household adoption of indigenous and modern knowledge in rangeland management interventions in Northern Kenya.

### Dataset Source
The dataset is described via a Croissant schema available at the link below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print main metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

Let's print out the record sets and their main field `@id`s, as understood by `mlcroissant`.

In [ ]:
# List all available record sets (by @id)
print("Available record sets:")
for rs in dataset.record_sets:
    print(f"  Record set name: {rs.name}, @id: {rs.id}")
    for field in rs.fields:
        print(f"    Field: {field.name}, @id: {field.id}")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames. We use the record set and field `@id`s found in the overview step above.

*All references to record sets and fields use their Croissant `@id`.*

In [ ]:
# --- Discover record sets and select for analysis ---
# Re-run code if you want to get current record_set @ids and names
record_sets = [rs.id for rs in dataset.record_sets]
print(f"Found record sets: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")

# As an example, let's list the columns and show the head of the first available record set
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nColumns in first record set ({first_record_set}):")
    print(dataframes[first_record_set].columns.tolist())
    dataframes[first_record_set].head()
else:
    print("No record sets were discovered in the dataset.")

## 4. Exploratory Data Analysis (EDA)

Here we illustrate typical data processing steps:
- Filtering records by numeric criteria
- Normalizing a numeric field
- Grouping (aggregating) by a categorical field

**Note:** Fill in `numeric_field_id` and `group_field_id` with actual values shown in the field lists above.

In [ ]:
# --- Select fields for EDA (replace these @ids as appropriate) ---
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Fields available in record set {record_set_id}:")
    print(df.columns.tolist())

    # Example: try to guess a numeric field if possible, otherwise set manually
    numeric_candidates = [c for c in df.columns if 'loglikelihood' in c.lower() or 'coef' in c.lower() or 'value' in c.lower() or df[c].dtype.kind in 'fi']
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = df.columns[0]  # Fallback

    print(f"Selected numeric field for analysis: {numeric_field_id}")

    # Filtering (example: greater than 0.1)
    try:
        threshold = 0.1
        filtered_df = df[df[numeric_field_id].astype(float) > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field_id].astype(float).mean()
        std = filtered_df[numeric_field_id].astype(float).std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - mean) / std
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field
        # Attempt to pick a categorical field
        cat_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        if cat_candidates:
            group_field_id = cat_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (showing first few rows):")
            print(grouped_df.head())
        else:
            print("No obvious categorical group field found.")

    except Exception as e:
        print(f"Error during EDA: {e}")
else:
    print("No record sets to perform EDA.")

## 5. Visualization
Visualize data distributions and relationships. Below, we'll plot a histogram of the selected numeric field and, if possible, a boxplot grouped by the selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if it exists
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Could not plot: required fields missing.")

## 6. Conclusion

We have demonstrated how to use the `mlcroissant` library to load, explore, and process datasets described by Croissant schemas. Using entity `@id`s for referencing, we:
 - Loaded metadata and discovered record sets
 - Loaded records from record sets using their `@id`
 - Explored key numeric and categorical fields
 - Visualized attribute distributions

You can extend this notebook by selecting other record sets or fields (by their `@id`) for further, domain-specific analyses.